# Imports

In [ ]:
import json
import glob
import math
import os
import random
import time
from datetime import datetime, timezone
import collections
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

#from google.colab import drive

In [ ]:
#for colab version
#drive.mount("/content/drive")

file_path = "/content/drive/My Drive/ml-training/merged"
checkpoint_path = "/content/drive/My Drive/ml-training/checkpoints"
sessions_path = "/content/drive/My Drive/ml-training/sessions.json"
root_path = "/content/drive/My Drive/ml-training"

#local version
#drive.mount("/content/drive")

file_path = "merged"
checkpoint_path = "checkpoints"
sessions_path = "sessions.json"
root_path = "."

In [ ]:
print("this is a test")

# Session Reconstructor

In [ ]:
DATA_DIR = file_path
OUTPUT = sessions_path

def parse_ts(ts_str):

  return datetime.fromisoformat(ts_str)

def dt_ms(t1, t2):

  """Milliseconds from t1 to t2."""
  return max(0, int((t2 - t1).total_seconds() * 1000))

def classify_outcome(events):

  """Derive session outcome from the terminal message."""
  msgs = [e["message_name"] for e in events]
  if any("HANDOVER_REQUIRED" in m for m in msgs):
    if any("HANDOVER_PREPARATION_FAILURE" in m for m in msgs):
        return "ho_failure"
    if any("PATH_SWITCH" in m for m in msgs):
        return "ho_success_x2"
  if any("UE_CONTEXT_RELEASE_COMPLETE" in m for m in msgs):
    cause = next(
      (e.get("cause") for e in reversed(events) if e.get("cause")), None
    )
    if cause:
      # Flatten cause dict to a readable string
      if isinstance(cause, dict):
          cause_str = cause.get("label") or cause.get("value") or str(cause)
      else:
          cause_str = str(cause)

      return f"release_{cause_str}"

    return "normal_release"

  if any("UE_CONTEXT_RELEASE_COMMAND" in m for m in msgs):

    return "partial_release"  # missing complete — spans file boundary

  return "unknown"

def split_by_boundary(evts):
  """
  Split a list of events for one (enb_id, mme_id) pair into individual
  sessions. A new session starts on INITIAL_UE_MESSAGE or when a
  RELEASE_COMPLETE closes the previous one.
  """
  sessions = []
  current = []
  for e in evts:
    msg = e["message_name"]
    if "INITIAL_UE_MESSAGE" in msg and current:
      sessions.append(current)
      current = []
    current.append(e)
    if "UE_CONTEXT_RELEASE_COMPLETE" in msg:
      sessions.append(current)
      current = []
  if current:
    sessions.append(current)

  return sessions

def build_session(evts, session_idx):

  """Convert a raw event list into a structured session dict."""
  evts = sorted(evts, key=lambda e: e["timestamp"])
  t0 = parse_ts(evts[0]["timestamp"])

  structured_events = []
  prev_t = t0
  for e in evts:
    t = parse_ts(e["timestamp"])
    structured_events.append(
      {
        "message_name": e["message_name"],
        "direction": e["direction"],
        "dt_ms": dt_ms(prev_t, t),
        "cell_id": e["cell_id"],
      }
    )
    prev_t = t

  outcome = classify_outcome(evts)
  duration = dt_ms(t0, parse_ts(evts[-1]["timestamp"]))
  cell_id = evts[0]["cell_id"]
  enb_id = evts[0].get("enb_ue_s1ap_id")
  mme_id = evts[0].get("mme_ue_s1ap_id")

  return {
    "session_id": session_idx,
    "enb_ue_s1ap_id": enb_id,
    "mme_ue_s1ap_id": mme_id,
    "cell_id": cell_id,
    "start_time": evts[0]["timestamp"],
    "duration_ms": duration,
    "n_events": len(evts),
    "outcome": outcome,
    "events": structured_events,
  }

def main():

  files = sorted(glob.glob(f"{DATA_DIR}/*.json"))
  print(f"Loading {len(files)} trace files...")

  # Load all events sorted by timestamp across all files
  all_s1_events = []
  for fpath in files:
    with open(fpath) as f:
      records = json.load(f)
    # Some trace files store a single record as a top-level dict instead of
    # a list of records. Iterating a dict yields its string keys, and
    # r["interface"] on one of those keys is what raises "string indices
    # must be integers, not str" — wrap it in a list so it's handled the
    # same as the rest.
    if isinstance(records, dict):
      records = [records]
    for r in records:
      if not isinstance(r, dict):
        print(f"  Skipping malformed record in {fpath}: {type(r).__name__}")
        continue
      if r.get("interface") == "S1":
          all_s1_events.append(r)

  all_s1_events.sort(key=lambda e: e["timestamp"])
  print(f"Total S1AP events: {len(all_s1_events)}")

  # Group by (enb_ue_s1ap_id, mme_ue_s1ap_id)
  buckets = collections.defaultdict(list)
  unlinked = []
  for e in all_s1_events:
    eid = e.get("enb_ue_s1ap_id")
    mid = e.get("mme_ue_s1ap_id")
    if eid is None and mid is None:
      unlinked.append(e)
    else:
      buckets[(eid, mid)].append(e)

  print(f"Unique (enb_id, mme_id) buckets: {len(buckets)}")
  print(f"Unlinked events (no IDs): {len(unlinked)}")

  sessions = []
  for key, evts in buckets.items():
    for sess_evts in split_by_boundary(evts):
      sessions.append(build_session(sess_evts, len(sessions)))

  # Group unlinked events by time proximity (within 200ms) and cell_id.
  # This captures HO pairs: HANDOVER_REQUIRED + PREPARATION_FAILURE arrive
  # 10–30ms apart but both have no S1AP IDs in this trace format.
  unlinked.sort(key=lambda e: (e["cell_id"], e["timestamp"]))
  unlinked_groups = []
  current_group = []
  PROXIMITY_MS = 200

  for e in unlinked:
    if not current_group:
      current_group.append(e)
    else:
      prev = current_group[-1]
      same_cell = (e["cell_id"] == prev["cell_id"])
      dt = (parse_ts(e["timestamp"]) - parse_ts(prev["timestamp"])).total_seconds() * 1000
      if same_cell and dt <= PROXIMITY_MS:
        current_group.append(e)
      else:
        unlinked_groups.append(current_group)
        current_group = [e]
  if current_group:
    unlinked_groups.append(current_group)

  for grp in unlinked_groups:
    sessions.append(build_session(grp, len(sessions)))

  print(f"Total sessions reconstructed: {len(sessions)}")

  # Summary stats
  outcomes = collections.Counter(s["outcome"] for s in sessions)
  print("\nOutcome distribution:")
  for outcome, cnt in outcomes.most_common():
    print(f"  {outcome:30s} {cnt:5d}  ({cnt/len(sessions)*100:.1f}%)")

  lengths = [s["n_events"] for s in sessions]
  print(f"\nSession length (events): min={min(lengths)} median={sorted(lengths)[len(lengths)//2]} max={max(lengths)}")

  cells = collections.Counter(s["cell_id"] for s in sessions)
  print("\nSessions per cell:")
  for cell, cnt in sorted(cells.items()):
    print(f"  cell_{cell}: {cnt}")

  with open(OUTPUT, "w") as f:
    json.dump(sessions, f)
  print(f"\nSaved {len(sessions)} sessions to {OUTPUT}")

if __name__ == "__main__":

    main()

# State Machine

In [ ]:
# S1AP states
S_IDLE           = "IDLE"
S_CONNECTING     = "CONNECTING"       # after INITIAL_UE_MESSAGE
S_CONTEXT_ACTIVE = "CONTEXT_ACTIVE"   # after INITIAL_CONTEXT_SETUP_RESPONSE
S_RELEASING      = "RELEASING"        # after UE_CONTEXT_RELEASE_COMMAND
S_HO_PENDING     = "HO_PENDING"       # after HANDOVER_REQUIRED
S_DONE           = "DONE"             # terminal

# Message name fragments used for pattern matching
_TRANSITIONS = {
  S_IDLE: [
    ("S1_INITIAL_UE_MESSAGE",                S_CONNECTING),
    ("S1_UE_CONTEXT_RELEASE_COMMAND",         S_RELEASING),  # partial session
  ],
  S_CONNECTING: [
    ("S1_UPLINK_NAS_TRANSPORT",               S_CONNECTING),
    ("S1_DOWNLINK_NAS_TRANSPORT",             S_CONNECTING),
    ("S1_INITIAL_CONTEXT_SETUP_REQUEST",      S_CONNECTING),
    ("S1_INITIAL_CONTEXT_SETUP_RESPONSE",     S_CONTEXT_ACTIVE),
    ("S1_UE_CONTEXT_RELEASE_COMMAND",         S_RELEASING),
    ("S1_LOCATION_REPORTING_CONTROL",         S_CONNECTING),
    ("S1_LOCATION_REPORT",                    S_CONNECTING),
    ("S1_UE_CAPABILITY_INDICATION",           S_CONNECTING),
    ("S1_CELL_TRAFFIC_TRACE",                 S_CONNECTING),
    ("S1_MME_STATUS_TRANSFER",                S_CONNECTING),
  ],
  S_CONTEXT_ACTIVE: [
    ("S1_UPLINK_NAS_TRANSPORT",               S_CONTEXT_ACTIVE),
    ("S1_DOWNLINK_NAS_TRANSPORT",             S_CONTEXT_ACTIVE),
    ("S1_ERAB_SETUP_REQUEST",                 S_CONTEXT_ACTIVE),
    ("S1_ERAB_SETUP_RESPONSE",                S_CONTEXT_ACTIVE),
    ("S1_ERAB_RELEASE_COMMAND",               S_CONTEXT_ACTIVE),
    ("S1_ERAB_RELEASE_RESPONSE",              S_CONTEXT_ACTIVE),
    ("S1_UE_CAPABILITY_INDICATION",           S_CONTEXT_ACTIVE),
    ("S1_LOCATION_REPORTING_CONTROL",         S_CONTEXT_ACTIVE),
    ("S1_LOCATION_REPORT",                    S_CONTEXT_ACTIVE),
    ("S1_CELL_TRAFFIC_TRACE",                 S_CONTEXT_ACTIVE),
    ("S1_HANDOVER_REQUIRED",                  S_HO_PENDING),
    ("S1_PATH_SWITCH_REQUEST",                S_CONTEXT_ACTIVE),
    ("S1_PATH_SWITCH_REQUEST_ACKNOWLEDGE",    S_CONTEXT_ACTIVE),
    ("S1_UE_CONTEXT_RELEASE_REQUEST",         S_RELEASING),
    ("S1_UE_CONTEXT_RELEASE_COMMAND",         S_RELEASING),
    ("S1_MME_STATUS_TRANSFER",                S_CONTEXT_ACTIVE),
  ],
  S_HO_PENDING: [
    ("S1_HANDOVER_PREPARATION_FAILURE",       S_CONTEXT_ACTIVE),
    ("S1_UE_CONTEXT_RELEASE_COMMAND",         S_RELEASING),
    ("S1_DOWNLINK_NAS_TRANSPORT",             S_HO_PENDING),
  ],
  S_RELEASING: [
    ("S1_UE_CONTEXT_RELEASE_COMPLETE",        S_DONE),
    # Allow stray messages that arrive before release completes
    ("S1_UPLINK_NAS_TRANSPORT",               S_RELEASING),
    ("S1_LOCATION_REPORT",                    S_RELEASING),
  ],
  S_DONE: [],
}

# Pre-index: state -> list of valid next message names
VALID_NEXT = {state: [msg for msg, _ in txns] for state, txns in _TRANSITIONS.items()}

# Pre-index: (state, message_fragment) -> next_state
_STATE_MAP = {}
for state, txns in _TRANSITIONS.items():
  for msg, next_state in txns:
    _STATE_MAP[(state, msg)] = next_state

def next_state(current_state, message_name):

  """Return the next state given current state and observed message name."""
  # Exact match first
  key = (current_state, message_name)
  if key in _STATE_MAP:
      return _STATE_MAP[key]
  # Partial match (message_name may have prefix/suffix noise)
  for (state, msg), nxt in _STATE_MAP.items():
      if state == current_state and (msg in message_name or message_name in msg):
          return nxt
  # Unknown transition: stay in current state rather than crash
  return current_state

def valid_next_messages(current_state):

  """Return list of valid next message names from current state."""
  return VALID_NEXT.get(current_state, [])

def session_state_trace(session):

  """
  Walk a session's events and return the state at each step.
  Useful for validating reconstructed sessions and diagnosing issues.
  """
  state = S_IDLE
  trace = [state]
  for evt in session["events"]:
    state = next_state(state, evt["message_name"])
    trace.append(state)
  return trace


def validate_session(session):

  """Returns True if all transitions in the session are valid."""
  state = S_IDLE
  for evt in session["events"]:
    valid = valid_next_messages(state)
    msg = evt["message_name"]
    matched = any(v in msg or msg in v for v in valid)
    if not matched:
        return False, f"Invalid: {msg} from state {state}"
    state = next_state(state, msg)
  return True, "ok"


# Tokenize

In [ ]:
SESSIONS_FILE = sessions_path
OUT_DIR       = root_path

# Time delta bucket boundaries in milliseconds.
# Bucket index = position in this list where value falls.
# 20-100ms previously held ~47% of all events in one bucket; split at 40ms
# (the natural break point in this dataset's dt distribution) for resolution
# where most of the data actually lives.
TIME_BUCKETS = [0, 5, 20, 40, 100, 500, 2000, 10000]

SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]


def dt_bucket(dt_ms):

  for i, boundary in enumerate(TIME_BUCKETS):
    if dt_ms <= boundary:
      return i

  return len(TIME_BUCKETS)

def session_to_tokens(session):
  """
  Convert one session to a list of string tokens:
    [CELL_{id}, <BOS>, MSG|DIR|T{bucket}, ..., <EOS>]
  """
  tokens = [f"CELL_{session['cell_id']}", "<BOS>"]
  for evt in session["events"]:
    direction = "UL" if evt["direction"] == 2 else "DL"
    bucket = dt_bucket(evt["dt_ms"])
    tokens.append(f"{evt['message_name']}|{direction}|T{bucket}")
  tokens.append("<EOS>")

  return tokens


def build_vocab(all_token_seqs):

  vocab = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
  for seq in all_token_seqs:
    for tok in seq:
      if tok not in vocab:
        vocab[tok] = len(vocab)
  return vocab


def encode(token_seq, vocab):

  unk = vocab["<UNK>"]
  return [vocab.get(tok, unk) for tok in token_seq]


def main():

  with open(SESSIONS_FILE) as f:
    sessions = json.load(f)

  # Sort temporally for correct split
  sessions.sort(key=lambda s: s["start_time"])

  # Filter to sessions with at least 2 events (single-event partial sessions
  # are only useful for arrival modeling, not sequence modeling)
  sessions = [s for s in sessions if s["n_events"] >= 2]
  print(f"Sessions with >=2 events: {len(sessions)}")

  # Drop file-boundary artifacts: "unknown" and "partial_release" sessions
  # are cut off mid-protocol (the trace window ended before the session
  # did), not real session endings. Appending <EOS> to them would teach
  # the model that almost any state can validly end a session.
  before = len(sessions)
  sessions = [s for s in sessions if s["outcome"] not in ("unknown", "partial_release")]
  print(f"Sessions after dropping unknown/partial_release: {len(sessions)} "
        f"(dropped {before - len(sessions)})")

  # Temporal split: 80 / 10 / 10
  n = len(sessions)
  n_train = int(n * 0.80)
  n_val   = int(n * 0.90)
  splits = {
    "train": sessions[:n_train],
    "val":   sessions[n_train:n_val],
    "test":  sessions[n_val:],
  }
  print(f"Split: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")

  # Tokenize
  tokenized = {split: [session_to_tokens(s) for s in slist]
                for split, slist in splits.items()}

  # Build vocab from training set only
  vocab = build_vocab(tokenized["train"])
  print(f"Vocabulary size: {len(vocab)} tokens")

  # Token frequency in training set
  freq = collections.Counter()
  for seq in tokenized["train"]:
    freq.update(seq)
  print("\nTop 20 most frequent tokens:")
  for tok, cnt in freq.most_common(20):
    print(f"  {tok:60s} {cnt:6d}")

  # Encode all splits
  encoded = {split: [encode(seq, vocab) for seq in seqs]
              for split, seqs in tokenized.items()}

  # Write outputs
  with open(f"{OUT_DIR}/vocab.json", "w") as f:
    json.dump(vocab, f, indent=2)

  for split, seqs in encoded.items():
    with open(f"{OUT_DIR}/{split}_tokens.json", "w") as f:
      json.dump(seqs, f)

  # Outcome label per session, parallel to {split}_tokens.json, so training
  # can report perplexity broken down by outcome (e.g. normal vs ho_failure).
  for split, slist in splits.items():
    with open(f"{OUT_DIR}/{split}_outcomes.json", "w") as f:
      json.dump([s["outcome"] for s in slist], f)

  # Also write a human-readable sample
  sample_path = f"{OUT_DIR}/token_samples.txt"
  with open(sample_path, "w") as f:
    f.write("=== Sample tokenized sessions (first 10, train set) ===\n\n")
    for i, (sess, toks) in enumerate(zip(splits["train"][:10], tokenized["train"][:10])):
      f.write(f"Session {sess['session_id']} | cell={sess['cell_id']} | "
              f"outcome={sess['outcome']} | {sess['n_events']} events\n")
      f.write("  " + " -> ".join(toks) + "\n\n")

  print(f"\nWrote vocab.json, train/val/test_tokens.json, token_samples.txt to {OUT_DIR}")

if __name__ == "__main__":

  main()

# The model logic

In [ ]:
class CausalSelfAttention(nn.Module):

  def __init__(self, d_model, n_heads, max_len, dropout):

    super().__init__()
    assert d_model % n_heads == 0

    self.n_heads = n_heads
    self.head_dim = d_model // n_heads
    self.qkv = nn.Linear(d_model , 3 * d_model, bias=False)
    self.proj = nn.Linear(d_model, d_model, bias=False)
    self.dropout = nn.Dropout(dropout)
    mask = torch.triu(torch.ones(max_len, max_len), diagonal=1).bool()
    self.register_buffer("causal_mask", mask)

  def forward(self, x):

    B, T, C = x.shape
    q, k, v = self.qkv(x).split(C, dim=2)
    q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    scale = math.sqrt(self.head_dim)
    attn = (q @ k.transpose(-2, -1)) / scale
    attn = attn.masked_fill(self.causal_mask[:T, :T], float("-inf"))
    attn = self.dropout(F.softmax(attn, dim=-1))
    out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)

    return self.proj(out)


In [ ]:
class TransformerBlock(nn.Module):

  def __init__(self, d_model, n_heads, max_len, dropout):

    super().__init__()

    self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
    self.ff = nn.Sequential(
        nn.Linear(d_model, 4 * d_model),
        nn.GELU(),
        nn.Linear(4 * d_model, d_model),
        nn.Dropout(dropout),
    )
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def forward(self, x):

        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))

        return x

In [ ]:
class SessionTransformer(nn.Module):

  def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, max_len=64, dropout=0.1):

    super().__init__()

    self.max_len = max_len
    self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
    self.pos_emb = nn.Embedding(max_len, d_model)
    self.drop = nn.Dropout(dropout)
    self.blocks = nn.ModuleList(
        [TransformerBlock(d_model, n_heads, max_len, dropout) for _ in range(n_layers)]
    )
    self.ln_f = nn.LayerNorm(d_model)
    self.head = nn.Linear(d_model, vocab_size, bias=False)
    self._init_weights()

  def _init_weights(self):

    for module in self.modules():
      if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, std=0.02)
      elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, std=0.02)

  def forward(self, idx):

    B, T = idx.shape
    assert T <= self.max_len, f"Sequence length {T} exceeds max_len {self.max_len}"
    pos = torch.arange(T, device=idx.device).unsqueeze(0)
    x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
    for block in self.blocks:
        x = block(x)
    x = self.ln_f(x)
    return self.head(x)

  @torch.no_grad()
  def generate(self, context, vocab, inv_vocab, state_machine=None, max_new_tokens=30, temperature=1.0, top_k=10):

    self.eval()
    device = next(self.parameters()).device
    pad = vocab.get("<PAD>", 0)
    eos = vocab.get("<EOS>", -1)

    idx = torch.tensor([[vocab.get(t, vocab["<UNK>"]) for t in context]], dtype=torch.long, device=device)

    # Walk context tokens to derive current SM state before generating
    sm_state = None
    if state_machine is not None:
      sm_state = S_IDLE
      for ctx_tok in context:
        # Skip special/cell tokens — only protocol messages advance the SM
        if ctx_tok.startswith("CELL_") or ctx_tok in ("<BOS>", "<EOS>", "<PAD>", "<UNK>"):
            continue
        msg = ctx_tok.split("|")[0]
        sm_state = next_state(sm_state, msg)

    generated = list(context)
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.max_len:]
      logits = self(idx_cond)[:, -1, :]  # (1, vocab_size)

      # Apply state machine mask if provided
      if state_machine is not None and sm_state is not None:
        valid_msgs = valid_next_messages(sm_state)
        # Always allow EOS so generation can terminate cleanly
        mask = torch.full((logits.shape[-1],), float("-inf"), device=device)
        mask[eos] = 0.0
        for msg in valid_msgs:
          for tok, idx_val in vocab.items():
            # Match by substring (tokens have |DIR|T{n} suffix)
            if msg in tok or tok.split("|")[0] in msg:
              mask[idx_val] = 0.0
        logits = logits + mask

      logits = logits / temperature
      if top_k > 0:
        topk_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < topk_vals[:, -1:]] = float("-inf")

      probs = F.softmax(logits, dim=-1)
      next_idx = torch.multinomial(probs, 1)
      next_tok = inv_vocab.get(next_idx.item(), "<UNK>")

      generated.append(next_tok)
      idx = torch.cat([idx, next_idx], dim=1)

      if state_machine is not None and sm_state is not None:
        msg = next_tok.split("|")[0]
        sm_state = next_state(sm_state, msg)

      if next_idx.item() == eos:
        break

    return generated

In [ ]:
def count_parameters(model):

  return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Train

In [ ]:
DATA_DIR = file_path
CKPT_DIR = checkpoint_path
os.makedirs(CKPT_DIR, exist_ok=True)

# Hyperparameters — tuned for ~13k sessions on a laptop (CPU-friendly)
CFG = {
  "d_model":    128,
  "n_heads":    4,
  "n_layers":   4,
  "max_len":    64,
  "dropout":    0.1,
  "batch_size": 64,
  "lr":         3e-4,
  "epochs":     30,
  "grad_clip":  1.0,
  "seed":       42,
}

class SessionDataset(Dataset):

  def __init__(self, token_seqs, max_len):

    self.seqs = []
    for seq in token_seqs:
      # Truncate to max_len; short sequences are fine
      self.seqs.append(seq[:max_len])

  def __len__(self):

    return len(self.seqs)

  def __getitem__(self, idx):

    return torch.tensor(self.seqs[idx], dtype=torch.long)


def collate_fn(batch, pad_idx=0):

  """Pad sequences in a batch to the same length."""
  max_len = max(len(s) for s in batch)
  padded = torch.zeros(len(batch), max_len, dtype=torch.long)
  for i, s in enumerate(batch):
      padded[i, :len(s)] = s

  return padded

def compute_loss(model, batch, pad_idx=0):
  """
  Next-token prediction loss.
  Input:  batch[:, :-1]
  Target: batch[:, 1:]
  Ignores padding positions.
  """

  x = batch[:, :-1]
  y = batch[:, 1:]
  logits = model(x)
  B, T, V = logits.shape
  loss = nn.functional.cross_entropy(
      logits.reshape(B * T, V),
      y.reshape(B * T),
      ignore_index=pad_idx,
  )

  return loss

def evaluate(model, loader, device, pad_idx=0):

  model.eval()
  total_loss = 0.0
  n_batches = 0
  with torch.no_grad():
    for batch in loader:
      batch = batch.to(device)
      total_loss += compute_loss(model, batch, pad_idx).item()
      n_batches += 1

  return total_loss / max(n_batches, 1)

def main():
  torch.manual_seed(CFG["seed"])
  random.seed(CFG["seed"])

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Device: {device}")

  # Load data
  with open(f"{root_path}/vocab.json") as f:
    vocab = json.load(f)
  pad_idx = vocab["<PAD>"]
  vocab_size = len(vocab)
  print(f"Vocab size: {vocab_size}")

  splits = {}
  for split in ("train", "val", "test"):
    with open(f"{root_path}/{split}_tokens.json") as f:
      splits[split] = json.load(f)
  print(f"Data: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")

  datasets = {
    split: SessionDataset(seqs, CFG["max_len"])
    for split, seqs in splits.items()
  }
  loaders = {
    "train": DataLoader(datasets["train"], batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate_fn),
    "val":   DataLoader(datasets["val"],   batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn),
    "test":  DataLoader(datasets["test"],  batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn),
  }

  model = SessionTransformer(
    vocab_size=vocab_size,
    d_model=CFG["d_model"],
    n_heads=CFG["n_heads"],
    n_layers=CFG["n_layers"],
    max_len=CFG["max_len"],
    dropout=CFG["dropout"],
  ).to(device)
  print(f"Model parameters: {count_parameters(model):,}")

  optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["epochs"], eta_min=CFG["lr"] / 10
  )

  log = []
  best_val_loss = float("inf")

  for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for batch in loaders["train"]:
      batch = batch.to(device)
      optimizer.zero_grad()
      loss = compute_loss(model, batch, pad_idx)
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
      optimizer.step()
      epoch_loss += loss.item()

    scheduler.step()
    train_loss = epoch_loss / len(loaders["train"])
    val_loss   = evaluate(model, loaders["val"], device, pad_idx)
    elapsed    = time.time() - t0

    # Perplexity is the natural metric for sequence models
    train_ppl = math.exp(min(train_loss, 20))
    val_ppl   = math.exp(min(val_loss, 20))

    print(f"Epoch {epoch:3d}/{CFG['epochs']} | "
          f"train_loss={train_loss:.4f} ppl={train_ppl:.2f} | "
          f"val_loss={val_loss:.4f} ppl={val_ppl:.2f} | "
          f"lr={scheduler.get_last_lr()[0]:.2e} | {elapsed:.1f}s")

    log.append({
      "epoch": epoch,
      "train_loss": train_loss, "train_ppl": train_ppl,
      "val_loss": val_loss,     "val_ppl":   val_ppl,
    })

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      torch.save({"model_state": model.state_dict(), "cfg": CFG,
                  "vocab_size": vocab_size, "epoch": epoch},
                  f"{CKPT_DIR}/best_model.pt")
      print(f"  -> saved best model (val_loss={val_loss:.4f})")

  torch.save({"model_state": model.state_dict(), "cfg": CFG,
              "vocab_size": vocab_size, "epoch": CFG["epochs"]},
              f"{CKPT_DIR}/final_model.pt")

  test_loss = evaluate(model, loaders["test"], device, pad_idx)
  test_ppl  = math.exp(min(test_loss, 20))
  print(f"\nTest loss={test_loss:.4f} ppl={test_ppl:.2f}")

  # The temporal split concentrates rare outcomes (e.g. ho_failure) in the
  # test set, so the aggregate test_ppl above mixes "normal traffic"
  # generalization with "anomaly" perplexity. Break it down by outcome.
  test_by_outcome = {}
  with open(f"{root_path}/test_outcomes.json") as f:
    test_outcomes = json.load(f)

  by_outcome_seqs = collections.defaultdict(list)
  for seq, outcome in zip(splits["test"], test_outcomes):
    by_outcome_seqs[outcome].append(seq)

  print("\nTest perplexity by outcome:")
  for outcome, seqs in sorted(by_outcome_seqs.items()):
    ds = SessionDataset(seqs, CFG["max_len"])
    loader = DataLoader(ds, batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn)
    loss = evaluate(model, loader, device, pad_idx)
    ppl  = math.exp(min(loss, 20))
    test_by_outcome[outcome] = {"n": len(seqs), "loss": loss, "ppl": ppl}
    print(f"  {outcome:30s} n={len(seqs):4d}  loss={loss:.4f}  ppl={ppl:.2f}")

  with open(f"{DATA_DIR}/training_log.json", "w") as f:
      json.dump({"config": CFG, "log": log,
                  "test_loss": test_loss, "test_ppl": test_ppl,
                  "test_by_outcome": test_by_outcome}, f, indent=2)
  print(f"Saved training_log.json and checkpoints to {CKPT_DIR}/")

if __name__ == "__main__":

    main()